# CAUKER: Causal-Kernel Generation

CAUKER (Xie et al., 2025) generates synthetic **time series for classification
pre-training** by combining Gaussian Process (GP) kernel composition with
Structural Causal Models (SCM) in five steps:

1. sample $K \sim U(1, K_{\max})$ kernels from a bank of 36 variants;
2. compose them with random $+$ / $\times$ operations;
3. draw root-node signals from GP(mean, composite kernel) priors;
4. sample activation functions for SCM edges;
5. propagate signals through a random DAG with edge activations.

Reference: Xie, S., et al. (2025). *CAUKER*. arXiv:2508.02879v3, Algorithm 1.

In [ ]:
import os
import sys

import numpy as np

sys.path.append(os.path.abspath(os.path.join("..", "..")))

from s2generator.scm import CaukerPipeline

In [ ]:
rng = np.random.RandomState(0)
pipe = CaukerPipeline(target_length=256)

## Generate a single time series

``generate`` returns a series of shape $(d, L)$ ($d$ observed variables, $L$ time
steps). With ``input_dimension=1`` it produces a univariate series, matching the
CAUKER pre-training setup.

In [ ]:
x = pipe.generate(rng, n_inputs_points=256, input_dimension=1)
print("univariate shape:", x.shape)

## Multivariate series

In [ ]:
x = pipe.generate(rng, n_inputs_points=256, input_dimension=3)
print("multivariate shape:", x.shape)

## Classification label

CAUKER is designed for classification pre-training. Passing ``n_classes`` returns
$(x, y)$, where the label is derived from the series summary statistic. For
balanced labels use ``generate_batch`` below.

In [ ]:
x, y = pipe.generate(rng, n_inputs_points=256, input_dimension=1, n_classes=5)
print("series shape:", x.shape, " label:", y)

## Balanced labeled batch

In [ ]:
batch = pipe.generate_batch(
    rng, n_samples=40, n_inputs_points=128, input_dimension=1, n_classes=4
)
labels = [lab for _, lab in batch]
print("n samples:", len(batch), " sample shape:", batch[0][0].shape)
print("class counts:", np.bincount(labels))

## Custom DAG

A user-supplied adjacency matrix fixes the causal graph; here a chain over 6 nodes.

In [ ]:
V = 6
adj = np.zeros((V, V), dtype=int)
for i in range(V - 1):
    adj[i, i + 1] = 1

x = pipe.generate(rng, n_inputs_points=256, input_dimension=2, adjacency=adj)
print("shape with custom graph:", x.shape)

## GP banks

The pipeline exposes the kernel and mean-function banks it samples from.

In [ ]:
print("n kernels:", pipe.n_kernels, " n mean functions:", pipe.n_mean_functions)

## Visualize generated sequences

One line plot per sequence: each variate is drawn as a line over time. The cell
below generates a small batch of multivariate series and shows each in its own
subplot.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# Generate a few multivariate series (3 variates each) to inspect.
rng = np.random.RandomState(0)
seqs = [pipe.generate(rng, n_inputs_points=128, input_dimension=3) for _ in range(4)]

n = len(seqs)
ncols = 2
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3 * nrows), squeeze=False)
for i, s in enumerate(seqs):
    ax = axes[i // ncols][i % ncols]
    ax.plot(s.T)  # (d, L) -> (L, d): one line per variate
    ax.set_title(f"series {i}  ({s.shape[0]} variates)")
    ax.set_xlabel("time step")
for j in range(n, nrows * ncols):
    axes[j // ncols][j % ncols].axis("off")
fig.tight_layout()
plt.show()